# 02 · 統一監控面板

把專案各階段的監控數據集中在這裡看：**① BPE tokenizer ② 訓練 loss ③ 資料品質**。
資料來源都是 `artifacts/` 裡的報表/CSV，所以你在終端機跑完（`make bpe` / `make train` / `make data`），回這裡 Restart & Run All 就是最新狀態。

> 監控原則：每個區塊都「先表格（可逐行審核）+ 後圖表（看趨勢）」。

In [ ]:
import sys, json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
ART = ROOT / 'artifacts'
print('artifacts:', ART)
pd.set_option('display.max_rows', 30)

## ① BPE tokenizer 監控

來源：`artifacts/bpe_merges.csv`（每步合併）+ `bpe_report.json`（彙總）。先跑過 `make bpe`。

In [ ]:
bpe_csv = ART / 'bpe_merges.csv'
assert bpe_csv.exists(), '先在終端機跑 `make bpe`'
merges = pd.read_csv(bpe_csv)
rep = json.loads((ART / 'bpe_report.json').read_text())
print(f"vocab {rep['base_vocab']} -> {rep['final_vocab']}　|　"
      f"{rep['orig_chars']:,} 字元 -> {rep['final_seq_len']:,} token　|　"
      f"每 token {rep['chars_per_token']} 字元")
# 審核表：前 30 步合併（可逐行看它學了什麼）
merges.head(30)

In [ ]:
# 三張監控圖：序列壓縮 / 合併頻率(邊際效益) / vocab 成長
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
ax[0].plot(merges['step'], merges['seq_len']); ax[0].set_title('序列長度（越合併越短=壓縮）')
ax[0].set_xlabel('合併步數'); ax[0].set_ylabel('token 數')
ax[1].plot(merges['step'], merges['freq'], color='indianred'); ax[1].set_title('每步合併頻率（邊際效益遞減）')
ax[1].set_xlabel('合併步數'); ax[1].set_ylabel('該配對出現次數')
ax[2].plot(merges['step'], merges['vocab_size'], color='green'); ax[2].set_title('vocab 成長（線性）')
ax[2].set_xlabel('合併步數'); ax[2].set_ylabel('vocab size')
plt.tight_layout(); plt.show()
print('學到的詞片段（最長）:', rep['learned_fragments_top25'][:12])
print(f"樣本 {rep['sample_text']!r}: char {rep['sample_char_len']} -> BPE {rep['sample_bpe_len']}: {rep['sample_tokens']}")

## ② 訓練 loss 監控

來源：`artifacts/runs/*.csv`（每個 run 一個）。跑過 `make train` 或 `--run_name xxx` 就會出現。
實線=val、虛線=train；兩線拉開=過擬合。

In [ ]:
runs = sorted((ART / 'runs').glob('*.csv')) if (ART / 'runs').exists() else []
if not runs:
    print('還沒有訓練紀錄，先跑 make train')
else:
    plt.figure(figsize=(9, 4))
    summary = []
    for p in runs:
        d = pd.read_csv(p)
        line, = plt.plot(d['step'], d['val_loss'], '-', label=f'{p.stem} val')
        plt.plot(d['step'], d['train_loss'], '--', color=line.get_color(), alpha=0.5)
        gap = d['val_loss'].iloc[-1] - d['train_loss'].iloc[-1]
        summary.append({'run': p.stem, 'best_val': round(d['val_loss'].min(), 4),
                        'final_gap(過擬合)': round(gap, 3)})
    plt.xlabel('step'); plt.ylabel('loss'); plt.legend(); plt.grid(alpha=0.3)
    plt.title('訓練 loss（多 run 比較）'); plt.tight_layout(); plt.show()
    display(pd.DataFrame(summary))

## ③ 資料品質監控

來源：`artifacts/data_report.json`（清洗/去重各砍多少）+ `src/data/stats.py`（量/熵/壓縮比）。

In [ ]:
dr = ART / 'data_report.json'
if not dr.exists():
    print('還沒有資料報表，先跑 make data 或 make data-demo')
else:
    report = json.loads(dr.read_text())
    stages = pd.DataFrame(report['stages'])[['stage', 'in', 'out', 'dropped']]
    display(stages)
    plt.figure(figsize=(7, 3))
    plt.bar(stages['stage'], stages['dropped'], color='indianred')
    plt.title('各階段丟掉的文件數'); plt.ylabel('丟幾篇'); plt.tight_layout(); plt.show()
    try:
        from src.data import stats
        m = stats.pipeline_metrics(ART)
        print(f"字元 {m['total_chars']:,}　熵 {m['char_entropy']} bits　壓縮比 {m['compression_ratio']}　重複率 {m['dup_rate']}")
    except Exception as e:
        print('品質指標略過:', e)

## ④ char vs BPE tokenizer 對照

跟 `char_tok` / `bpe_tok` 兩個 run 比。

> ⚠️ **重點：raw val loss 跨 tokenizer 不能直接比**（char 從 65 類猜、bpe 從 365 類猜，難度不同）。
> 公平指標是 **BPC（bits per character）**：把 loss 換算成「每個字元幾 bit」，越低越好。

In [ ]:
import math
comp = []
for name in ['char_tok', 'bpe_tok']:
    mp = ART / 'runs' / f'{name}.meta.json'
    cp = ART / 'runs' / f'{name}.csv'
    if not (mp.exists() and cp.exists()):
        continue
    m = json.loads(mp.read_text())
    d = pd.read_csv(cp)
    best = d['val_loss'].min()
    bpc = best / math.log(2) / m['chars_per_token']
    comp.append({'tokenizer': m['tokenizer'], 'vocab': m['vocab_size'],
                 'chars/token': m['chars_per_token'],
                 'best_val_loss (不可比)': round(best, 3),
                 'BPC bits/char (可比越低越好)': round(bpc, 3)})
display(pd.DataFrame(comp))
print('raw loss 跨 tokenizer 不可比；BPC 才公平。BPE 用一半的序列長度達到相當甚至更低的 BPC。')
for fn, label in [('gen_char', 'char'), ('gen_bpe', 'bpe')]:
    p = ART / f'{fn}.txt'
    if p.exists():
        print(f'\n=== {label} 生成樣本 ===')
        print(p.read_text()[:350])

In [ ]:
# 三格對比圖：raw loss(誤導) vs BPC(公平) vs 效率
import math
labs, raw, bpc, cpt = [], [], [], []
for name in ['char_tok', 'bpe_tok']:
    m = json.loads((ART/'runs'/f'{name}.meta.json').read_text())
    d = pd.read_csv(ART/'runs'/f'{name}.csv')
    best = d['val_loss'].min()
    labs.append(m['tokenizer']); raw.append(best)
    bpc.append(best/math.log(2)/m['chars_per_token']); cpt.append(m['chars_per_token'])
colors = ['#4c72b0', '#dd8452']
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
ax[0].bar(labs, raw, color=colors); ax[0].set_title('raw val loss（不可比，會誤導）\nBPE 看起來輸')
ax[1].bar(labs, bpc, color=colors); ax[1].set_title('BPC bits/char（公平，越低越好）\nBPE 其實贏')
ax[2].bar(labs, cpt, color=colors); ax[2].set_title('每 token 字元數（越高越省序列）\nBPE 序列半長')
for a, vals in zip(ax, [raw, bpc, cpt]):
    for i, v in enumerate(vals):
        a.text(i, v, f'{v:.2f}', ha='center', va='bottom')
plt.tight_layout(); plt.show()
print('教訓：天真看 raw loss 會誤判 BPE 較爛；歸一化成 BPC 才看出 BPE 更好、且只用一半序列。')

In [ ]:
# val loss 曲線「換成 BPC 單位」→ 跨 tokenizer 公平比（這樣 BPE 才不會看起來較差）
import math
plt.figure(figsize=(8, 4))
for name in ['char_tok', 'bpe_tok']:
    m = json.loads((ART/'runs'/f'{name}.meta.json').read_text())
    d = pd.read_csv(ART/'runs'/f'{name}.csv')
    bpc = d['val_loss'] / math.log(2) / m['chars_per_token']
    plt.plot(d['step'], bpc, marker='o', ms=3, label=f"{m['tokenizer']}")
plt.xlabel('訓練步數'); plt.ylabel('val BPC（bits/char，越低越好）')
plt.title('把 val loss 換成 BPC 來呈現 → BPE 沒有比較差（甚至略低）')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
print('原則：同 tokenizer 比 raw loss OK；跨 tokenizer 一定要換成 BPC 這種「每字元」單位才公平。')

## ⑤ 單元測試健康狀態

在面板裡直接跑整套單元測試，把「程式碼還健不健康」也納入監控。綠=全過、紅=有失敗。

In [ ]:
# 用 subprocess 跑整套測試（等同 make test），把結果監控起來
import subprocess, re
p = subprocess.run(['python', '-m', 'unittest', 'discover', '-s', 'tests'],
                   cwd=str(ROOT), capture_output=True, text=True)
out = p.stderr  # unittest 把結果印到 stderr
m = re.search(r'Ran (\d+) tests', out)
total = int(m.group(1)) if m else 0
ok = 'OK' in out.splitlines()[-1] if out.strip() else False
fm = re.search(r'failures=(\d+)', out); em = re.search(r'errors=(\d+)', out)
failed = (int(fm.group(1)) if fm else 0) + (int(em.group(1)) if em else 0)
passed = total - failed
print(('✅ 全部通過' if ok else f'❌ {failed} 個失敗') + f'  —  {passed}/{total} 個測試')
plt.figure(figsize=(6, 1.8))
plt.barh(['通過', '失敗'], [passed, failed], color=['seagreen', 'indianred'])
plt.title(f'單元測試健康狀態（{passed}/{total} 通過）'); plt.xlabel('測試數')
plt.tight_layout(); plt.show()
if not ok:
    print(out[-800:])

---
用法：終端機跑 `make bpe` / `make train` / `make data` 產生/更新數據 → 回這裡 **Kernel → Restart & Run All** 就是最新監控狀態。